# Debug edit behavior with ReflectionRegenerationPipeline

This notebook runs image editing through `qwen_latent_cot/inference/pipeline.py`.
It helps diagnose why refined outputs look like blur-only changes.


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

from qwen_latent_cot.inference.pipeline import ReflectionRegenerationPipeline
from qwen_latent_cot.models.qwen_image_backend import LocalQwenImageBackend
from qwen_latent_cot.models.reflector import HeuristicReflector


In [ ]:
# ===== paths and generation settings =====
MODEL_PATH = "/data1/weights/Qwen-Image-Edit"
PROMPT = "Several hands with warm, rich skin tones gently hold teacups around a wooden table, bathed in soft, golden lighting."
OUT_DIR = Path("outputs/notebook_debug")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
NUM_STEPS = 50
TRUE_CFG = 4.0

backend = LocalQwenImageBackend(
    model_path=MODEL_PATH,
    dtype="bfloat16",
    aspect_ratio="1:1",
)


In [ ]:
# ===== run 1: default (heuristic reflector) =====
pipeline_default = ReflectionRegenerationPipeline(
    image_backend=backend,
    reflector=HeuristicReflector(),
)

result_default = pipeline_default.run_and_save(
    prompt=PROMPT,
    output_dir=str(OUT_DIR / "default"),
    num_inference_steps=NUM_STEPS,
    guidance_scale=TRUE_CFG,
    seed=SEED,
    init_image=None,
)

result_default


In [ ]:
# ===== run 2: force stronger edit feedback =====
class FixedReflector:
    def __init__(self, reflection_text: str):
        self.reflection_text = reflection_text

    def reflect(self, prompt: str, image: Image.Image) -> str:
        del prompt, image
        return self.reflection_text

strong_reflection = (
    "<|refl_start|>"
    "<problem>The draft is soft and lacks local details, especially hand texture and table grain.</problem>"
    "<fix>Increase local sharpness and micro-contrast on hands and wooden table texture, while preserving global composition and lighting.</fix>"
    "<|refl_end|>"
)

pipeline_strong = ReflectionRegenerationPipeline(
    image_backend=backend,
    reflector=FixedReflector(strong_reflection),
)

result_strong = pipeline_strong.run_and_save(
    prompt=PROMPT,
    output_dir=str(OUT_DIR / "strong_feedback"),
    num_inference_steps=NUM_STEPS,
    guidance_scale=TRUE_CFG,
    seed=SEED,
    init_image=None,
)

result_strong


In [ ]:
# ===== visualize and inspect debug files =====
draft_default = Image.open(result_default["draft"]).convert("RGB")
refined_default = Image.open(result_default["refined"]).convert("RGB")
draft_strong = Image.open(result_strong["draft"]).convert("RGB")
refined_strong = Image.open(result_strong["refined"]).convert("RGB")

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes[0, 0].imshow(draft_default); axes[0, 0].set_title("default / draft"); axes[0, 0].axis("off")
axes[0, 1].imshow(refined_default); axes[0, 1].set_title("default / refined"); axes[0, 1].axis("off")
axes[1, 0].imshow(draft_strong); axes[1, 0].set_title("strong / draft"); axes[1, 0].axis("off")
axes[1, 1].imshow(refined_strong); axes[1, 1].set_title("strong / refined"); axes[1, 1].axis("off")
plt.tight_layout()

print("default debug:", result_default.get("refine_debug"))
print("strong  debug:", result_strong.get("refine_debug"))
